#  Loop 

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import datetime
import itertools
import warnings
warnings.filterwarnings("ignore")
import zipfile
import pandas_datareader.data as web
import os
import yfinance as yf

In [ ]:
start_date = datetime.datetime(1970, 1, 1)
end_date = datetime.datetime.now()

## Grid config

In [ ]:
DRAWDOWN_THRESHOLDS = [0.03, 0.05, 0.08, 0.10, 0.12, 0.15]

# ← seul paramètre à changer pour chaque marché : "SP500", "DAX", "Nikkei"
MARKET = "SP500"

MARKETS_CFG = {
    #            ff cols          mkt ETF col      finance ETF col (fallback PatroCorr)
    "SP500":  {"ff_finance_col": 10, "ff_market_col": 0, "sectors_mkt_col": "SPY",     "sectors_fin_col": "XLF"},
    "DAX":    {"ff_finance_col": 10, "ff_market_col": 0, "sectors_mkt_col": "EXSA.DE", "sectors_fin_col": "EXV1.DE"},
    "Nikkei":    {"ff_finance_col": 10, "ff_market_col": 0, "sectors_mkt_col": "1615.T",    "sectors_fin_col": "1617.T"},
    "EuroStoxx": {"ff_finance_col": 10, "ff_market_col": 0, "sectors_mkt_col": "^STOXX50E", "sectors_fin_col": "EXX1.DE"},
    "FTSE100":    {"ff_finance_col": 10, "ff_market_col": 0, "sectors_mkt_col": "ISF.L",      "sectors_fin_col": "IUKF.L"},
    "CAC40":      {"ff_finance_col": 10, "ff_market_col": 0, "sectors_mkt_col": "EXSA.DE",    "sectors_fin_col": "EXV1.DE"},
    "ASX200":     {"ff_finance_col": 10, "ff_market_col": 0, "sectors_mkt_col": "STW.AX",     "sectors_fin_col": "OZF.AX"},
    "China":      {"ff_finance_col": 10, "ff_market_col": 0, "sectors_mkt_col": "MCHI",       "sectors_fin_col": "CHIE"},
}
market_cfg = MARKETS_CFG[MARKET]

IND_COL   = MARKET
data_path = f"data/{MARKET}/"

experiment = [[5, "D"], [10, "D"], [15, "D"], [4, "W"], [8, "W"], [12, "W"], [16, "W"], [6, "M"], [12, "M"], [18, "M"], [24, "M"]]

# pandas 2.2+ uses "ME" for month-end; older versions use "M"
_ME = "ME" if tuple(int(x) for x in pd.__version__.split(".")[:2]) >= (2, 2) else "M"
FREQ_MAP = {"D": "D", "W": "W-FRI", "M": _ME}

start_date = datetime.datetime(1970, 1, 1)
end_date   = datetime.datetime.now()

## Functions from indicators notebook

In [ ]:
def resample_data(df_daily, freq, price_col):
    """Resample the daily dataframe to the requested frequency.

    The trailing dropna removes calendar-day rows introduced by
    .resample(); without it, daily-frequency configurations would
    silently include weekends with returns=0."""
    agg = {price_col: "last", "returns": "sum"}
    return df_daily.resample(freq).agg(agg).dropna(subset=[price_col])


def compute_crisis_target(df, ret_col, horizon, drawdown):
    """Binary crisis target.

    The forward cumulative simple return over `horizon` periods is
    compared to `-drawdown`: a row is flagged when the sum of the
    next `horizon` returns falls below the negative threshold.

    `df` must already be at the target frequency.  At weekly frequency,
    horizon=4 means 4 weeks; at monthly frequency, horizon=12 means 12
    months.  The last `horizon` rows have an unknown future and are
    returned as NaN, so that the downstream dropna() drops them from
    the regression sample."""
    fut_cum = df[ret_col].rolling(window=horizon).sum().shift(-horizon)
    flag = (fut_cum <= -drawdown).astype("Int64")
    flag[fut_cum.isna()] = pd.NA
    return flag


def fit_logit_indicator(data, indicator, target, add_const=True):
    X = data[[indicator]].copy()
    y = data[target].copy()
    if add_const:
        X = sm.add_constant(X)
        
    result = sm.Logit(y, X).fit(disp=0, method='bfgs', maxiter=200)

    # Check perfect separation
    if result.mle_retvals.get('warnflag', 0) != 0:
        print(f"    WARNING convergence issue for {indicator}")

    return {
        "indicator":      indicator,
        "model":          result,
        "coef":           result.params[indicator],
        "p_value":        result.pvalues[indicator],
        "z_score":        result.tvalues[indicator],
        "pseudo_r2":      result.prsquared,
        "odds_ratio":     np.exp(result.params[indicator]),
        "predicted_prob": result.predict(X),
    }


def plot_indicator(df, indicator_col, crisis_col, sp500_col=IND_COL):
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax1.plot(df.index, np.log(df[sp500_col]), "navy", lw=2, label="S&P 500 (log)")
    ax1.set_ylabel("S&P 500 (log)", color="navy")
    ax1.grid(True, alpha=0.3)
    crisis_starts = df[df[crisis_col].diff() == 1].index
    period_width  = df.index[1] - df.index[0] if len(df) > 1 else pd.Timedelta(7, "D")
    for date in crisis_starts:
        ax1.axvspan(date, date + period_width,
                    facecolor="red", alpha=0.4,
                    label="Crisis" if date == crisis_starts[0] else "")
    ax2 = ax1.twinx()
    ax2.plot(df.index, df[indicator_col], "darkred", lw=2.5)
    ax2.set_ylabel(indicator_col, color="darkred")
    ax1.legend(loc="upper left")
    plt.title(indicator_col)
    plt.tight_layout()
    return fig




def get_fama_french_local(file_name):
    df = pd.read_csv(f"{file_name}.csv", index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index)
    return df

## Data preparation

In [ ]:
# ── Main index → df_daily on the native trading-day grid ───────────
index_raw   = pd.read_csv(f"{data_path}index.csv", index_col=0, parse_dates=True).sort_index()
df_daily    = index_raw[["Close"]].copy()
df_daily    = df_daily.rename(columns={"Close": IND_COL})
df_daily["returns"] = df_daily[IND_COL].pct_change()

# ── Indicateurs bruts ─────────────────────────────────────────────────────────
hy_spread_raw = pd.read_csv(f"{data_path}hy_spread.csv",    index_col=0, parse_dates=True)
yc_raw        = pd.read_csv(f"{data_path}yield_curve.csv",  index_col=0, parse_dates=True)
vix_raw       = pd.read_csv(f"{data_path}vix.csv",          index_col=0, parse_dates=True)
m2_raw        = pd.read_csv(f"{data_path}m2.csv",           index_col=0, parse_dates=True)
gdp_raw       = pd.read_csv(f"{data_path}gdp.csv",          index_col=0, parse_dates=True)
unemp_raw     = pd.read_csv(f"{data_path}unrate.csv",       index_col=0, parse_dates=True)
# Shiller CAPE ou Euro CAPE Proxy selon le marché
if os.path.exists(f"{data_path}shiller_cape.csv"):
    cape_raw = pd.read_csv(f"{data_path}shiller_cape.csv", index_col=0, parse_dates=True)
    cape_raw.columns = ["CAPE"]
    euro_cape_mode = False
else:
    # EuroStoxx : CAPE proxy = prix réel / MA 10 ans prix réel (via CPI)
    cpi_raw        = pd.read_csv(f"{data_path}cpi.csv", index_col=0, parse_dates=True)
    cpi_raw.columns = ["CPI"]
    euro_cape_mode = True
    cape_raw       = None  # calculé par fréquence dans la boucle principale
sectors_raw   = pd.read_csv(f"{data_path}sectors.csv",      index_col=0, parse_dates=True)

# ── Règle de Sahm ─────────────────────────────────────────────────────────────
unemp_raw["U3MAvg"]        = unemp_raw["UNRATE"].rolling(window=3).mean()
unemp_raw["ULow12M"]       = unemp_raw["U3MAvg"].rolling(window=12).min()
unemp_raw["SahmIndicator"] = unemp_raw["U3MAvg"] - unemp_raw["ULow12M"]

# ── M2 croissance YoY ─────────────────────────────────────────────────────────
m2_raw["M2_Growth"] = m2_raw["M2"].pct_change(12) * 100

# ── Corrélation systémique journalière ────────────────────────────────────────
mkt_col             = market_cfg["sectors_mkt_col"]
# Keep only columns with ≥2000 non-null rows (avoid short-history ETFs killing the sample)
_min_rows     = 2000
_valid_cols   = [c for c in sectors_raw.columns if sectors_raw[c].notna().sum() >= _min_rows]
if mkt_col not in _valid_cols:                    # always keep market column
    _valid_cols = [mkt_col] + _valid_cols
sectors_clean = sectors_raw[_valid_cols].ffill().dropna(subset=[mkt_col])
sec_returns   = sectors_clean.pct_change()
mkt_ret       = sec_returns[mkt_col].dropna()
sectors_ret   = sec_returns.drop(columns=[mkt_col], errors="ignore")
systemic_corr_daily = sectors_ret.rolling(window=60).corr(mkt_ret).mean(axis=1)

# ── Corrélation Patro (Finance / Marché, rolling 36M) ────────────────────────
# Primaire : Fama-French industry file
# Fallback  : ETF secteur finance vs ETF marché (sectors.csv)
if os.path.exists(f"{data_path}ff_industry.csv"):
    df_ind     = get_fama_french_local(f"{data_path}ff_industry")
    df_fac     = get_fama_french_local(f"{data_path}ff_factors")
    df_patro_m = pd.DataFrame({
        "FinanceRet": df_ind.iloc[:, market_cfg["ff_finance_col"]].astype(float),
        "MarketRet":  df_fac.iloc[:, market_cfg["ff_market_col"]].astype(float),
    }).loc[start_date:end_date]
    print("PatroCorr : Fama-French industry")
else:
    fin_col = market_cfg["sectors_fin_col"]
    # If the finance ETF is all-NaN (delisted), pick the first available non-market column
    if fin_col not in sectors_raw.columns or sectors_raw[fin_col].isna().all():
        available = [c for c in sectors_clean.columns if c != mkt_col]
        fin_col   = available[0] if available else mkt_col
        print(f"PatroCorr : finance ETF manquant → fallback colonne '{fin_col}'")
    _sec_m     = sectors_raw[[fin_col, mkt_col]].dropna(how="all").resample(_ME).last().pct_change().dropna()
    df_patro_m = pd.DataFrame({
        "FinanceRet": _sec_m[fin_col],
        "MarketRet":  _sec_m[mkt_col],
    })
    print(f"PatroCorr : fallback sectors ({fin_col} vs {mkt_col})")

df_patro_m["PatroCorr"] = (
    df_patro_m["FinanceRet"]
    .rolling(window=36, min_periods=12)
    .corr(df_patro_m["MarketRet"])
)
daily_idx  = pd.date_range(start=df_patro_m.index[0], end=df_patro_m.index[-1], freq="D")
df_patro_m = df_patro_m.reindex(daily_idx).ffill()

# ── Signal de Neely (MA 252 jours) ───────────────────────────────────────────
neely_ma_daily     = df_daily[IND_COL].rolling(252).mean()
neely_signal_daily = (df_daily[IND_COL] < neely_ma_daily).astype(int)
neely_signal_daily.name = "Neely_252D"

print(f"[{MARKET}] df_daily      : {df_daily.index.min().date()} → {df_daily.index.max().date()} ({len(df_daily)} jours)")
print(f"hy_spread_raw : {len(hy_spread_raw)} lignes  |  col = {list(hy_spread_raw.columns)}")
print(f"vix_raw       : {len(vix_raw)} lignes  |  col = {list(vix_raw.columns)}")
print(f"sectors_raw   : {sectors_raw.shape}  |  mkt_col = {mkt_col}")
print(f"df_patro_m    : {df_patro_m.index.min().date()} → {df_patro_m.index.max().date()}")
print(f"neely bearish : {neely_signal_daily.sum()} périodes")

## Préparation des nouveaux indicateurs

In [ ]:
# ── Chargement des indicateurs supplémentaires ───────────────────────────────
# Tous les fichiers existent pour tous les marchés (proxys US quand pas d'équivalent local).
# Voir data_loading.ipynb pour le détail des sources par marché.

yc3m_raw   = pd.read_csv(f"{data_path}yield_curve_10y3m.csv",   index_col=0, parse_dates=True)
cfnai_raw  = pd.read_csv(f"{data_path}cfnai.csv",               index_col=0, parse_dates=True)

# ── CISS (ECB Composite Indicator of Systemic Stress) ─────────────────────────
ciss_raw = pd.read_csv(f"{data_path}ciss.csv", index_col=0, parse_dates=True)
if "CISS" not in ciss_raw.columns:
    ciss_raw.columns = ["CISS"]

# ── NFCI (Chicago Fed National Financial Conditions Index) ────────────────────
# Proxy du Growth-at-Risk (Adrian et al., 2019). Valeurs > 0 = stress financier.
# Série hebdomadaire FRED, disponible depuis 1971 pour les US.
# Utilisé comme proxy US pour tous les marchés (conditions financières mondiales).
nfci_raw = pd.read_csv(f"{data_path}nfci.csv", index_col=0, parse_dates=True)
if "NFCI" not in nfci_raw.columns:
    nfci_raw.columns = ["NFCI"]

# ── EPU (Economic Policy Uncertainty — Baker, Bloom & Davis) ─────────────────
# Indice mensuel (ou daily US) mesurant l'incertitude de politique économique
# via l'analyse de la presse. Valeurs élevées = forte incertitude.
epu_raw = pd.read_csv(f"{data_path}epu.csv", index_col=0, parse_dates=True)
if "EPU" not in epu_raw.columns:
    epu_raw.columns = ["EPU"]

# ── Transformations ───────────────────────────────────────────────────────────

# ── Indicateurs calculés sur df_daily (marché-agnostiques) ───────────────────
df_daily["RealVol_21D"] = df_daily["returns"].rolling(21).std() * np.sqrt(252) * 100
_gain = df_daily["returns"].clip(lower=0)
_loss = (-df_daily["returns"]).clip(lower=0)
_rs   = _gain.ewm(com=13, min_periods=14).mean() / _loss.ewm(com=13, min_periods=14).mean().replace(0, np.nan)
df_daily["RSI_14"] = 100 - (100 / (1 + _rs))

print(f"[{MARKET}] Indicateurs supplémentaires chargés.")

# ── Downside Beta (SRISK proxy) ──────────────────────────────────────────────
# Mesure la sensibilité du secteur financier aux baisses du marché (rendements < 0).
# Proxy du LRMES utilisé dans le calcul du SRISK (Brownlees & Engle, 2017).
# Utilise les mêmes données sectorielles que PatroCorr.
_fin_col_db = market_cfg["sectors_fin_col"]
_mkt_col_db = market_cfg["sectors_mkt_col"]

# Fallback si la colonne finance est absente
if _fin_col_db not in sectors_raw.columns or sectors_raw[_fin_col_db].isna().all():
    _available_db = [c for c in sectors_clean.columns if c != _mkt_col_db]
    _fin_col_db   = _available_db[0] if _available_db else _mkt_col_db
    print(f"DownsideBeta : finance ETF manquant → fallback colonne '{_fin_col_db}'")

# Rendements journaliers
_db_df = sectors_raw[[_fin_col_db, _mkt_col_db]].dropna(how="all").ffill().pct_change().dropna()
_db_fin_ret = _db_df[_fin_col_db]
_db_mkt_ret = _db_df[_mkt_col_db]

# Downside Beta rolling 252 jours (1 an) — uniquement les jours où le marché baisse
_window_db = 252
_min_obs_db = 63  # au moins ~3 mois de données

def _rolling_downside_beta(fin_ret, mkt_ret, window, min_obs):
    """Compute rolling downside beta: Cov(r_fin, r_mkt | r_mkt < 0) / Var(r_mkt | r_mkt < 0)."""
    ds_beta = pd.Series(np.nan, index=fin_ret.index)
    for i in range(window, len(fin_ret)):
        _f = fin_ret.iloc[i-window:i]
        _m = mkt_ret.iloc[i-window:i]
        mask = _m < 0
        if mask.sum() < min_obs:
            continue
        _f_down = _f[mask]
        _m_down = _m[mask]
        var_m = _m_down.var()
        if var_m > 0:
            ds_beta.iloc[i] = np.cov(_f_down, _m_down)[0, 1] / var_m
    return ds_beta

downside_beta_daily = _rolling_downside_beta(_db_fin_ret, _db_mkt_ret, _window_db, _min_obs_db)
downside_beta_daily.name = "Downside_Beta"

print(f"Downside_Beta : {downside_beta_daily.notna().sum()} non-null values "
      f"({downside_beta_daily.index.min().date()} → {downside_beta_daily.index.max().date()})")

print(f"NFCI          : {nfci_raw.shape[0]} rows "
      f"({nfci_raw.index.min().date()} → {nfci_raw.index.max().date()})")

print(f"EPU           : {epu_raw.shape[0]} rows "
      f"({epu_raw.index.min().date()} → {epu_raw.index.max().date()})")


# ── Realized Skewness (Zhang, He, Zhang & Wang, 2021) ────────────────────────
_rsk_w = 63
_r = df_daily["returns"]
_rm = _r.rolling(_rsk_w).mean()
_rs = _r.rolling(_rsk_w).std()
df_daily["RSkew"] = ((_r - _rm) ** 3).rolling(_rsk_w).mean() / (_rs ** 3)

# ── Realized Semivariance Ratio (Bollerslev, Li, Patton & Quaedvlieg, 2020) ──
_rsv_w = 63
_neg_sq = df_daily["returns"].clip(upper=0) ** 2
_tot_sq = df_daily["returns"] ** 2
df_daily["RSV_Ratio"] = _neg_sq.rolling(_rsv_w).sum() / _tot_sq.rolling(_rsv_w).sum().replace(0, np.nan)

# ── Hill Tail Estimator (Hill, 1975; Zhang & Chen, 2022) ─────────────────────
def _rolling_hill(ret, window=252, k=12):
    """Hill (1975) tail-index estimator with fixed `k = 12`.

    For a 252-day window with roughly half negative returns, this
    sits at the Hall-rule sweet spot `k ~ n^{2/5}`."""
    out = np.full(len(ret), np.nan)
    vals = ret.values
    for i in range(window, len(vals)):
        chunk = vals[i - window:i]
        chunk = chunk[~np.isnan(chunk)]
        losses = np.sort(-chunk[chunk < 0])[::-1]
        if k + 1 >= len(losses) or losses[k] <= 0:
            continue
        out[i] = np.mean(np.log(losses[:k] / losses[k]))
    return pd.Series(out, index=ret.index)

df_daily["HillTail"] = _rolling_hill(df_daily["returns"])

print(f"RSkew     : {df_daily['RSkew'].notna().sum()} non-null values")
print(f"RSV_Ratio : {df_daily['RSV_Ratio'].notna().sum()} non-null values")
print(f"HillTail  : {df_daily['HillTail'].notna().sum()} non-null values")


## Main sweep

In [ ]:
all_results = []

for DRAWDOWN_THRESHOLD in DRAWDOWN_THRESHOLDS:

    print(f"\n{'='*60}")
    print(f"  DRAWDOWN_THRESHOLD = {DRAWDOWN_THRESHOLD:.0%}")
    print(f"{'='*60}")

    for HORIZON, FREQ in experiment:

        PANDAS_FREQ = FREQ_MAP[FREQ]
        target_col  = f"Crisis_next_{HORIZON}{FREQ}"

        print(f"\n  [{FREQ} | horizon={HORIZON} | dd={DRAWDOWN_THRESHOLD:.0%}]")

        # Target is computed on the already-resampled dataframe, so
        # `horizon` carries the unit of FREQ (days, weeks, months).
        df = resample_data(df_daily, PANDAS_FREQ, IND_COL)
        df[target_col] = compute_crisis_target(
            df, ret_col="returns", horizon=HORIZON, drawdown=DRAWDOWN_THRESHOLD
        )
        df.dropna(subset=[target_col], inplace=True)

        df_all = df.copy()

        def _resample_s(s, freq, how="mean"):
            if freq == "D": return s
            return s.resample(freq).mean() if how == "mean" else s.resample(freq).last()

        # ── Indicateurs de base ────────────────────────────────────────────────
        df_all["HY_Spread"]    = _resample_s(hy_spread_raw["HY_SPREAD"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["Yield_Curve"]  = _resample_s(yc_raw["YIELD_CURVE"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["VIX"]          = _resample_s(vix_raw["VIX"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["M2_Growth"]    = _resample_s(m2_raw["M2_Growth"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["SahmIndicator"]= _resample_s(unemp_raw["SahmIndicator"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        if euro_cape_mode:
            # Euro CAPE Proxy : prix réel (déflaté CPI) / MA 10 ans prix réel
            cpi_s        = _resample_s(cpi_raw["CPI"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
            real_price   = df_all[IND_COL] / cpi_s
            w_cape       = {"W": 10*52, "M": 10*12, "D": 10*252}[FREQ]
            df_all["Shiller_CAPE"] = real_price / real_price.rolling(window=w_cape, min_periods=w_cape//4).mean()
        else:
            df_all["Shiller_CAPE"] = _resample_s(cape_raw["CAPE"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()

        df_all["GDP"]              = gdp_raw["GDP"].reindex(df_all.index, method="ffill")
        df_all["Buffett"]          = df_all[IND_COL] / df_all["GDP"]
        window_trend               = {"W": 52*10, "M": 12*10, "D": 252*10}[FREQ]
        # min_periods = 20% of window to handle markets with short history (e.g. China ~5 years)
        min_periods_trend          = max(window_trend // 5, 12)
        df_all["Trend10Y"]         = df_all["Buffett"].rolling(window=window_trend, min_periods=min_periods_trend).mean()
        df_all["BuffettDeviation"] = (df_all["Buffett"] - df_all["Trend10Y"]) / df_all["Trend10Y"]

        if PANDAS_FREQ == "D":
            df_all["Neely"]       = neely_signal_daily.reindex(df_all.index).ffill()
            df_all["PatroCorr"]   = df_patro_m["PatroCorr"].reindex(df_all.index).ffill()
            df_all["Systemic_Corr"] = systemic_corr_daily.reindex(df_all.index).ffill()
        else:
            df_all["Neely"]       = neely_signal_daily.resample(PANDAS_FREQ).last().reindex(df_all.index).ffill()
            df_all["PatroCorr"]   = df_patro_m["PatroCorr"].resample(PANDAS_FREQ).last().reindex(df_all.index).ffill()
            df_all["Systemic_Corr"] = systemic_corr_daily.resample(PANDAS_FREQ).last().reindex(df_all.index).ffill()


        # ── Indicateurs supplémentaires (mêmes pour tous les marchés) ─────────
        df_all["YC_10Y3M"]     = _resample_s(yc3m_raw.iloc[:, 0],   PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["CFNAI"]        = _resample_s(cfnai_raw.iloc[:, 0],  PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["CISS"]         = _resample_s(ciss_raw["CISS"],       PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["Downside_Beta"] = _resample_s(downside_beta_daily, PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["NFCI"]          = _resample_s(nfci_raw["NFCI"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["EPU"]           = _resample_s(epu_raw["EPU"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["RSkew"]         = _resample_s(df_daily["RSkew"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["RSV_Ratio"]     = _resample_s(df_daily["RSV_Ratio"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()
        df_all["HillTail"]      = _resample_s(df_daily["HillTail"], PANDAS_FREQ, "last").reindex(df_all.index).ffill()

        # ── Liste fixe — identique pour tous les marchés ───────────────────────
        indicator_cols = [
            "HY_Spread", "Yield_Curve", "Neely", "PatroCorr",
            "SahmIndicator", "VIX", "M2_Growth", "BuffettDeviation",
            "Systemic_Corr", "Shiller_CAPE",
            "YC_10Y3M",
            "CFNAI", "CISS", "Downside_Beta", "NFCI", "EPU",
            "RSkew", "RSV_Ratio", "HillTail",
        ]

        df_common = df_all.dropna(subset=indicator_cols + [target_col])
        n_obs     = len(df_common)
        n_crisis  = int(df_common[target_col].sum())

        if n_obs == 0:
            print("    WARNING: empty common window — skipping config")
            for col in indicator_cols:
                print(f"      {col:28s} {df_all[col].notna().sum()} non-null rows")
            continue

        print(f"    Common window: {df_common.index.min().date()} -> {df_common.index.max().date()} "
              f"| {n_obs} obs | {n_crisis} crises ({100*n_crisis/n_obs:.1f}%)")

        logit_results = {}

        def run(ind_col):
            try:
                res = fit_logit_indicator(data=df_common, indicator=ind_col, target=target_col)
                logit_results[ind_col] = res
                print(f"    {ind_col:28s} pseudo_r2={res['pseudo_r2']:.4f}  p={res['p_value']:.3f}")
                all_results.append({
                    "market":             MARKET,
                    "drawdown_threshold": DRAWDOWN_THRESHOLD,
                    "freq":               FREQ,
                    "horizon":            HORIZON,
                    "target":             target_col,
                    "indicator":          ind_col,
                    "pseudo_r2":          res["pseudo_r2"],
                    "p_value":            res["p_value"],
                    "z_score":            res["z_score"],
                    "coef":               res["coef"],
                    "odds_ratio":         res["odds_ratio"],
                    "n_obs":              n_obs,
                    "n_crisis":           n_crisis,
                })
            except Exception as e:
                print(f"    {ind_col}: ERROR — {e}")

        for col in indicator_cols:
            run(col)

        if logit_results:
            summary_table = pd.DataFrame({
                k: {"Pseudo_R2": v["pseudo_r2"], "P_Value": v["p_value"], "Z_Score": v["z_score"]}
                for k, v in logit_results.items()
            }).T.sort_values("Pseudo_R2", ascending=False)
            print(summary_table.round(4).to_string())

In [ ]:
df_all

## Results

In [ ]:
df_results = pd.DataFrame(all_results)
df_results.sort_values(["drawdown_threshold", "pseudo_r2"], ascending=[True, False], inplace=True)
df_results.reset_index(drop=True, inplace=True)
print(f"Total results: {len(df_results)} rows across {df_results['drawdown_threshold'].nunique()} thresholds")
df_results.head()

### Figure

In [ ]:
BG        = "#0f1117"
PANEL     = "#1a1d27"
ACCENT    = "#00d4aa"
GREY      = "#8892a4"
WHITE     = "#e8ecf4"
WARN      = "#ff6b6b"
CMAP_NAME = "YlGn"

indicators = sorted(df_results["indicator"].unique())

for DRAWDOWN_THRESHOLD in DRAWDOWN_THRESHOLDS:

    df_thresh = df_results[df_results["drawdown_threshold"] == DRAWDOWN_THRESHOLD].copy()
    if df_thresh.empty:
        print(f"No results for threshold {DRAWDOWN_THRESHOLD:.0%} — skipping")
        continue

    configs       = df_thresh[["freq", "horizon"]].drop_duplicates().sort_values(["freq", "horizon"])
    config_labels = [f"{r.freq}-{r.horizon}" for r in configs.itertuples()]

    # Pivot: rows = config, cols = indicator
    df_pivot = df_thresh.pivot_table(
        index=["freq", "horizon"], columns="indicator", values="pseudo_r2", aggfunc="first"
    ).reindex(columns=indicators)
    df_pivot.index = config_labels

    # P-value pivot
    df_pval = df_thresh.pivot_table(
        index=["freq", "horizon"], columns="indicator", values="p_value", aggfunc="first"
    ).reindex(columns=indicators)
    df_pval.index = config_labels

    n_configs = len(config_labels)
    n_inds    = len(indicators)

    fig = plt.figure(figsize=(22, 14), facecolor=BG)
    fig.suptitle(f"Indicator Sweep — Pseudo-R²  |  Drawdown threshold: {DRAWDOWN_THRESHOLD:.0%}",
                 fontsize=18, fontweight="bold", color=WHITE, y=0.97, fontfamily="monospace")

    gs = fig.add_gridspec(
        2, 2,
        height_ratios=[2.2, 1],
        width_ratios=[3, 1],
        hspace=0.45, wspace=0.08,
        left=0.07, right=0.97, top=0.92, bottom=0.07
    )

    ax_heat  = fig.add_subplot(gs[0, 0])
    ax_best  = fig.add_subplot(gs[0, 1])
    ax_bar   = fig.add_subplot(gs[1, 0])
    ax_table = fig.add_subplot(gs[1, 1])

    # ── 1. Heatmap (p-value) ──────────────────────────────────────────────────────
    import matplotlib.colors as mcolors

    pvals  = df_pval.values.astype(float)
    pseudo = df_pivot.values.astype(float)

    cmap = plt.get_cmap("RdYlGn_r")
    norm = mcolors.Normalize(vmin=0, vmax=0.1)
    im   = ax_heat.imshow(pvals, aspect="auto", cmap=cmap, norm=norm)

    ax_heat.set_xticks(range(n_inds))
    ax_heat.set_xticklabels(indicators, rotation=35, ha="right", fontsize=8.5,
                             color=WHITE, fontfamily="monospace")
    ax_heat.set_yticks(range(n_configs))
    ax_heat.set_yticklabels(config_labels, fontsize=9, color=WHITE, fontfamily="monospace")
    ax_heat.set_title("P-value by config × indicator  (darker = more significant)", color=GREY,
                       fontsize=10, pad=8, fontfamily="monospace")
    ax_heat.tick_params(colors=GREY, length=0)
    for spine in ax_heat.spines.values():
        spine.set_visible(False)
    ax_heat.set_facecolor(PANEL)

    for i in range(n_configs):
        for j in range(n_inds):
            p  = pvals[i, j]
            r2 = pseudo[i, j]
            if np.isnan(p):
                continue
            txt_color = "black" if p < 0.02 else WHITE
            sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
            ax_heat.text(j, i, f"{p:.3f}{sig}", ha="center", va="center",
                         fontsize=7, color=txt_color, fontfamily="monospace")

    cb = plt.colorbar(im, ax=ax_heat, fraction=0.03, pad=0.02)
    cb.ax.yaxis.set_tick_params(color=GREY, labelsize=7)
    cb.outline.set_visible(False)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color=GREY, fontfamily="monospace")
    ax_heat.text(1.08, -0.06, "*** p<0.01  ** p<0.05  * p<0.1", transform=ax_heat.transAxes,
                 fontsize=7, color=GREY, fontfamily="monospace")

    # ── 2. Best indicator per config ────────────────────────────────────────────
    ax_best.set_facecolor(PANEL)
    for spine in ax_best.spines.values():
        spine.set_visible(False)
    ax_best.set_title("Most significant\nper config", color=GREY, fontsize=10,
                       pad=8, fontfamily="monospace")
    ax_best.set_xlim(0, 1)
    ax_best.set_ylim(-0.5, n_configs - 0.5)
    ax_best.set_yticks(range(n_configs))
    ax_best.set_yticklabels(config_labels, fontsize=9, color=WHITE, fontfamily="monospace")
    ax_best.tick_params(left=False, bottom=False, labelbottom=False, colors=GREY, length=0)

    for i, cfg in enumerate(config_labels):
        row_p = df_pval.loc[cfg].dropna()
        if row_p.empty:
            continue
        best_ind = row_p.idxmin()
        best_p   = row_p.min()
        ax_best.barh(i, 0.85, color=ACCENT, alpha=0.15, height=0.6)
        ax_best.text(0.04, i, best_ind, va="center", fontsize=8,
                     color=ACCENT, fontfamily="monospace", fontweight="bold")
        ax_best.text(0.86, i, f"p={best_p:.3f}", va="center", ha="right",
                     fontsize=8, color=WHITE, fontfamily="monospace")

    # ── 3. Mean p-value per indicator ───────────────────────────────────────────
    ax_bar.set_facecolor(PANEL)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.spines["left"].set_color(GREY)
    ax_bar.spines["bottom"].set_color(GREY)

    means_p    = df_pval.mean(axis=0).sort_values(ascending=True)
    colors_bar = [ACCENT if i == 0 else "#2a9d8f" if i < 3 else "#3d5a80"
                  for i in range(len(means_p))]
    bars = ax_bar.bar(range(len(means_p)), means_p.values, color=colors_bar, width=0.65, zorder=2)
    ax_bar.axhline(0.05, color=WARN, linewidth=1, linestyle="--", zorder=3, label="p=0.05")
    ax_bar.set_xticks(range(len(means_p)))
    ax_bar.set_xticklabels(means_p.index, rotation=35, ha="right", fontsize=8.5,
                            color=WHITE, fontfamily="monospace")
    ax_bar.set_ylabel("Mean P-value", color=GREY, fontsize=9, fontfamily="monospace")
    ax_bar.set_title("Average P-value across all configs  (lower = better)", color=GREY,
                      fontsize=10, pad=8, fontfamily="monospace")
    ax_bar.tick_params(colors=GREY, length=3)
    ax_bar.yaxis.set_tick_params(labelcolor=GREY, labelsize=8)
    ax_bar.set_facecolor(PANEL)
    ax_bar.grid(axis="y", color=GREY, alpha=0.15, linewidth=0.6, zorder=0)
    ax_bar.legend(fontsize=7, labelcolor=WARN, framealpha=0, loc="upper left")

    for bar, val in zip(bars, means_p.values):
        ax_bar.text(bar.get_x() + bar.get_width() / 2, val + 0.001,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7,
                    color=WHITE, fontfamily="monospace")

    # ── 4. Top 3 per config ─────────────────────────────────────────────────────
    ax_table.set_facecolor(PANEL)
    ax_table.axis("off")
    ax_table.set_title("Most significant per config", color=GREY, fontsize=10,
                        pad=8, fontfamily="monospace")

    top3_rows = []
    for cfg in config_labels:
        row_p  = df_pval.loc[cfg].dropna().nsmallest(3)
        row_r2 = df_pivot.loc[cfg]
        for rank, (ind, pval) in enumerate(row_p.items(), 1):
            r2 = row_r2.get(ind, float("nan"))
            top3_rows.append([cfg, str(rank), ind, f"{pval:.3f}", f"{r2:.3f}"])

    if top3_rows:
        col_labels = ["Config", "#", "Indicator", "p-val", "R²"]
        tbl = ax_table.table(
            cellText=top3_rows,
            colLabels=col_labels,
            cellLoc="left",
            loc="upper left",
            bbox=[0, 0, 1, 1]
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(7.5)
        for (row_idx, col_idx), cell in tbl.get_celld().items():
            cell.set_facecolor(PANEL if row_idx > 0 else "#252836")
            cell.set_edgecolor("#2e3347")
            cell.set_text_props(
                color=ACCENT if (row_idx > 0 and col_idx == 1 and top3_rows[row_idx-1][1] == "1")
                      else WHITE,
                fontfamily="monospace",
                fontweight="bold" if row_idx == 0 else "normal"
            )

    # ── Save figure for this threshold ──────────────────────────────────────────
    os.makedirs(f"{data_path}results/", exist_ok=True)
    fname = f"{data_path}results/results_sweep_dd{int(DRAWDOWN_THRESHOLD*100):02d}pct.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    plt.close(fig)
    print(f"  → Saved: {fname}")

In [ ]:
df_results.to_csv(f"{data_path}results/results_sweep.csv", index=False)